# Lesson 28: Explainable AI: why did the model decide that? · Practice

**Open Fraud Labs Academy**

1. Copy your **practice key** from the Practice tab of this lesson on the Academy and paste it into the first code cell.
2. Run the first cell, then complete each exercise and run its check.
3. Every check that passes is saved to your lesson page automatically. Solve all three to unlock the quiz.

Tip: choose **File > Save a copy in Drive** to keep your work.

In [ ]:
# Step 1: paste your practice key between the quotes, then run this cell (Shift + Enter).
# Find your key on the lesson page, in the Practice tab.
OFL_KEY = ""

import pandas as pd, requests
DATA_URL = "https://raw.githubusercontent.com/Odugbile1993/openfraudlab-tiktok/main/data/loans.csv"
COURSE, LESSON = "data-science", 28

def ofl_check(exercise, check_code):
    """Runs the check for one exercise and, if it passes, records it on your Academy lesson page."""
    try:
        exec(check_code, globals())
    except AssertionError as e:
        print("Not quite yet:", e or "compare your result with the task and try again.")
        return
    except NameError as e:
        print("Something the task asks for is missing. Run your code cell first.", e)
        return
    print("Correct! Well done.")
    if not OFL_KEY.strip():
        print("Add your practice key in the first cell and run it, then run this check again to record your progress.")
        return
    try:
        r = requests.post("https://virxrqwxvgsbcnmhdwlp.supabase.co/rest/v1/rpc/practice_report",
                          headers={"apikey": "sb_publishable_8IenXF_4F5ew8hLai8RfVA_dNHq4I72", "Content-Type": "application/json"},
                          json={"p_key": OFL_KEY.strip(), "p_course": COURSE, "p_lesson": LESSON, "p_exercise": exercise}, timeout=20)
        d = r.json()
        if r.ok:
            print(f"Saved to your lesson page: {d['solved']} of {d['total']} exercises solved.")
        else:
            print("Could not save your progress:", d.get("message", r.text))
    except Exception as e:
        print("Could not reach the Academy to save your progress:", e)

print("Ready. Work through the exercises below." if OFL_KEY.strip() else "Paste your practice key above so your progress is saved.")

## Exercise 1: Read the coefficients

Fit the pipeline of StandardScaler and LogisticRegression(max_iter=1000, class_weight="balanced"). Store its coefficients as a Series indexed by feature name in coefs, and the name of the feature with the most negative coefficient in lowest.

<details><summary>Hint</summary>

logreg.coef_[0] holds one coefficient per feature. Wrap it in pd.Series(..., index=features), then .idxmin() names the smallest.

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

subs = pd.read_csv(DATA_URL.replace("loans.csv", "global/subscribers.csv"))
subs["avg_weekly_hours"] = subs["avg_weekly_hours"].fillna(subs["avg_weekly_hours"].median())
subs["monthly_contract"] = subs["contract"].map({"Annual": 0, "Monthly": 1})
features = ["age", "monthly_fee_usd", "tenure_months", "avg_weekly_hours",
            "devices", "support_tickets_90d", "auto_renew", "monthly_contract"]
X = subs[features]
y = subs["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000, class_weight="balanced"))
model.fit(X_train, y_train)
logreg = model.named_steps["logisticregression"]

coefs = ...
lowest = ...
print(coefs, lowest)

In [ ]:
# Check exercise 1 (run after your code above)
ofl_check(1, "_s = subs.copy() if \"subs\" in globals() and hasattr(subs, \"copy\") else None\nimport pandas as _pd\n_s = _pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/subscribers.csv\"))\n_s[\"avg_weekly_hours\"] = _s[\"avg_weekly_hours\"].fillna(_s[\"avg_weekly_hours\"].median())\n_s[\"monthly_contract\"] = _s[\"contract\"].map({\"Annual\": 0, \"Monthly\": 1})\n_base = [\"age\", \"monthly_fee_usd\", \"tenure_months\", \"avg_weekly_hours\", \"devices\", \"support_tickets_90d\", \"auto_renew\", \"monthly_contract\"]\nfrom sklearn.model_selection import train_test_split as _tts\nfrom sklearn.pipeline import make_pipeline as _mp\nfrom sklearn.preprocessing import StandardScaler as _SS\nfrom sklearn.linear_model import LogisticRegression as _LR\n_Xtr, _Xte, _ytr, _yte = _tts(_s[_base], _s[\"churned\"], test_size=0.2, random_state=42, stratify=_s[\"churned\"])\n_m = _mp(_SS(), _LR(max_iter=1000, class_weight=\"balanced\")).fit(_Xtr, _ytr)\n_c = _pd.Series(_m.named_steps[\"logisticregression\"].coef_[0], index=_base)\nassert isinstance(coefs, _pd.Series) and list(coefs.index) == _base, \"coefs should be a Series indexed by the feature names: pd.Series(logreg.coef_[0], index=features)\"\nassert ((coefs - _c).abs() < 1e-6).all(), \"coefs should hold the coefficients of the standardised, balanced logistic regression\"\nassert lowest == _c.idxmin(), \"lowest should be the feature with the most negative coefficient: coefs.idxmin()\"\n")

## Exercise 2: Importance by accuracy

Run permutation_importance on the test set with its default scoring (accuracy), n_repeats=10 and random_state=42. Store the mean drops as a Series indexed by feature name in imp, and the most important feature's name in top.

<details><summary>Hint</summary>

permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42). The means are in result.importances_mean.

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

subs = pd.read_csv(DATA_URL.replace("loans.csv", "global/subscribers.csv"))
subs["avg_weekly_hours"] = subs["avg_weekly_hours"].fillna(subs["avg_weekly_hours"].median())
subs["monthly_contract"] = subs["contract"].map({"Annual": 0, "Monthly": 1})
features = ["age", "monthly_fee_usd", "tenure_months", "avg_weekly_hours",
            "devices", "support_tickets_90d", "auto_renew", "monthly_contract"]
X = subs[features]
y = subs["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
from sklearn.inspection import permutation_importance

model = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000, class_weight="balanced"))
model.fit(X_train, y_train)

result = ...
imp = ...
top = ...
print(imp, top)

In [ ]:
# Check exercise 2 (run after your code above)
ofl_check(2, "_s = subs.copy() if \"subs\" in globals() and hasattr(subs, \"copy\") else None\nimport pandas as _pd\n_s = _pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/subscribers.csv\"))\n_s[\"avg_weekly_hours\"] = _s[\"avg_weekly_hours\"].fillna(_s[\"avg_weekly_hours\"].median())\n_s[\"monthly_contract\"] = _s[\"contract\"].map({\"Annual\": 0, \"Monthly\": 1})\n_base = [\"age\", \"monthly_fee_usd\", \"tenure_months\", \"avg_weekly_hours\", \"devices\", \"support_tickets_90d\", \"auto_renew\", \"monthly_contract\"]\nfrom sklearn.model_selection import train_test_split as _tts\nfrom sklearn.pipeline import make_pipeline as _mp\nfrom sklearn.preprocessing import StandardScaler as _SS\nfrom sklearn.linear_model import LogisticRegression as _LR\n_Xtr, _Xte, _ytr, _yte = _tts(_s[_base], _s[\"churned\"], test_size=0.2, random_state=42, stratify=_s[\"churned\"])\n_m = _mp(_SS(), _LR(max_iter=1000, class_weight=\"balanced\")).fit(_Xtr, _ytr)\nfrom sklearn.inspection import permutation_importance as _pi\n_r = _pd.Series(_pi(_m, _Xte, _yte, n_repeats=10, random_state=42).importances_mean, index=_base)\nassert isinstance(imp, _pd.Series) and list(imp.index) == _base, \"imp should be a Series indexed by feature name: pd.Series(result.importances_mean, index=features)\"\nassert ((imp - _r).abs() < 1e-9).all(), \"Use permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)\"\nassert top == _r.idxmax(), \"top should be the feature with the biggest mean drop: imp.idxmax()\"\n")

## Exercise 3: Explain one customer

Take the first test subscriber, row = X_test.iloc[[0]]. Compute each feature's contribution (coefficient × standardised value) as a Series contrib indexed by feature name. Then store the model's churn score for that subscriber (predict_proba, class 1) in p_churn.

<details><summary>Hint</summary>

pd.Series(logreg.coef_[0] * z, index=features). model.predict_proba(row) has columns [0, 1], so [0, 1] picks the churn score.

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

subs = pd.read_csv(DATA_URL.replace("loans.csv", "global/subscribers.csv"))
subs["avg_weekly_hours"] = subs["avg_weekly_hours"].fillna(subs["avg_weekly_hours"].median())
subs["monthly_contract"] = subs["contract"].map({"Annual": 0, "Monthly": 1})
features = ["age", "monthly_fee_usd", "tenure_months", "avg_weekly_hours",
            "devices", "support_tickets_90d", "auto_renew", "monthly_contract"]
X = subs[features]
y = subs["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000, class_weight="balanced"))
model.fit(X_train, y_train)
scaler = model.named_steps["standardscaler"]
logreg = model.named_steps["logisticregression"]
row = X_test.iloc[[0]]

z = scaler.transform(row)[0]
contrib = ...
p_churn = ...
print(contrib, p_churn)

In [ ]:
# Check exercise 3 (run after your code above)
ofl_check(3, "_s = subs.copy() if \"subs\" in globals() and hasattr(subs, \"copy\") else None\nimport pandas as _pd\n_s = _pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/subscribers.csv\"))\n_s[\"avg_weekly_hours\"] = _s[\"avg_weekly_hours\"].fillna(_s[\"avg_weekly_hours\"].median())\n_s[\"monthly_contract\"] = _s[\"contract\"].map({\"Annual\": 0, \"Monthly\": 1})\n_base = [\"age\", \"monthly_fee_usd\", \"tenure_months\", \"avg_weekly_hours\", \"devices\", \"support_tickets_90d\", \"auto_renew\", \"monthly_contract\"]\nfrom sklearn.model_selection import train_test_split as _tts\nfrom sklearn.pipeline import make_pipeline as _mp\nfrom sklearn.preprocessing import StandardScaler as _SS\nfrom sklearn.linear_model import LogisticRegression as _LR\n_Xtr, _Xte, _ytr, _yte = _tts(_s[_base], _s[\"churned\"], test_size=0.2, random_state=42, stratify=_s[\"churned\"])\n_m = _mp(_SS(), _LR(max_iter=1000, class_weight=\"balanced\")).fit(_Xtr, _ytr)\n_z = _m.named_steps[\"standardscaler\"].transform(_Xte.iloc[[0]])[0]\n_want = _pd.Series(_m.named_steps[\"logisticregression\"].coef_[0] * _z, index=_base)\nassert isinstance(contrib, _pd.Series) and list(contrib.index) == _base, \"contrib should be a Series indexed by feature name\"\nassert ((contrib - _want).abs() < 1e-6).all(), \"Each contribution is coefficient times standardised value: logreg.coef_[0] * z\"\nassert p_churn is not Ellipsis and abs(float(p_churn) - _m.predict_proba(_Xte.iloc[[0]])[0, 1]) < 1e-9, \"p_churn should be model.predict_proba(row)[0, 1]\"\n")

---
Done? Go back to the lesson on [openfraudlabs.com/academy](https://openfraudlabs.com/academy/lesson/?course=data-science&n=28) and take the quiz.